In [0]:
import dlt

In [0]:
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import *

@dlt.table(
    name = "bronze_orders",
    comment = "Raw data from the orders table in the database.",
    table_properties = {"quality": "bronze"}
)

def create_bronze_orders():
  return (
    spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "json")
      .option("cloudFiles.schemaLocation", "/Volumes/circuitbox/lakehouse/_schemas/bronze_orders_schema")
      .option("cloudFiles.inferColumnTypes", "true")
      .load('/Volumes/circuitbox/landing/operational_data/orders/')
      .withColumn("input_file_path", col("_metadata.file_path"))
      .withColumn("created_date", current_timestamp())
  )

customer_id
bigint
items: {"items": {"category": "string", "item_id": "bigint", "name": "string", "price": "bigint", "quantity": "bigint"}}
array
order_id
bigint
order_status
string
order_timestamp
string
payment_method
string
_rescued_data
string
input_file_path
string
created_date
timestamp

In [0]:
from pyspark.sql.functions import explode, col
from pyspark.sql.types import TimestampType

@dlt.table(
    name = "silver_orders",
    comment = "Data from the orders table from the bronze layer.",
    table_properties = {"quality": "silver"}
)
@dlt.expect_or_fail("valid_customer_id", "customer_id IS NOT NULL")
@dlt.expect_or_fail("valid_order_id", "order_id IS NOT NULL")
@dlt.expect("valid_order_status", "order_status IN ('Pending', 'Shipping', 'Cancelled', 'Completed')")
@dlt.expect("valid_payment_method", "payment_method IN ('Credit Card', 'PayPal', 'Bank Transfer')")

def create_silver_orders_clean():
    return (
        spark.readStream.table("LIVE.bronze_orders")
        .select(
            "customer_id",
            "order_id",
            "order_status",
            "order_timestamp",
            "payment_method",
            explode("items").alias("item"),
            "created_date"
        ).select(
            "customer_id",
            "order_id",
            "order_status",
            col("order_timestamp").cast(TimestampType()).alias("order_timestamp"),
            "payment_method",
            col("item.item_id").alias("item_id"),
            col("item.category").alias("item_category"),
            col("item.name").alias("item_name"),
            col("item.price").alias("item_price"),
            col("item.quantity").alias("item_quantity"),
            "created_date"
        )
        
    )